# Step 6: Pairwise Feature Engineering

## 1. Objective
The objective of this step is to transform each candidate pair from Step 5 into a numerical vector of features (similarity metrics, structured flags, semantic similarity, missingness indicators). This feature matrix will be used by the XGBoost classifier in Step 7 to classify each pair as a match (1) or non-match (0).

In [2]:
import pandas as pd
import numpy as np
import json
import sys
import os
from sklearn.model_selection import train_test_split

# Ensure project root is in sys.path regardless of execution directory
for p in ['.', '..', '../..', 'src']:
    abs_p = os.path.abspath(p)
    if abs_p not in sys.path:
        sys.path.append(abs_p)

from src.business_entity_resolution.features import generate_all_features


## 2. Load Candidate Pairs

In [4]:
cand_path = '../experiments/final_candidate_pairs_train.tsv' if os.path.exists('../experiments/final_candidate_pairs_train.tsv') else 'experiments/final_candidate_pairs_train.tsv'
candidates_df = pd.read_csv(cand_path, sep='\t')
print(f"Total candidate pairs: {len(candidates_df)}")


Total candidate pairs: 20000


## 3 & 4. Load Source Data & Normalized Data

In [6]:
base_data_dir = '../dataset/train' if os.path.exists('../dataset/train') else 'dataset/train'

s1_path = os.path.join(base_data_dir, 'train_source1.csv') if os.path.exists(os.path.join(base_data_dir, 'train_source1.csv')) else os.path.join(base_data_dir, 'train_source1.tsv')
s2_path = os.path.join(base_data_dir, 'train_source2.csv') if os.path.exists(os.path.join(base_data_dir, 'train_source2.csv')) else os.path.join(base_data_dir, 'train_source2.tsv')
s3_path = os.path.join(base_data_dir, 'train_source3.csv') if os.path.exists(os.path.join(base_data_dir, 'train_source3.csv')) else os.path.join(base_data_dir, 'train_source3.tsv')

s1 = pd.read_csv(s1_path, sep='\t' if s1_path.endswith('.tsv') else ',')
s2 = pd.read_csv(s2_path, sep='\t' if s2_path.endswith('.tsv') else ',')
s3 = pd.read_csv(s3_path, sep='\t' if s3_path.endswith('.tsv') else ',')

s1 = s1.add_prefix('s1_')
s1 = s1.rename(columns={'s1_entity_id': 'source1_entity_id', 's1_business_name': 's1_name', 's1_business_address': 's1_address'})

s2 = s2.add_prefix('cand_')
s2 = s2.rename(columns={'cand_entity_id': 'candidate_entity_id', 'cand_business_name': 'cand_name', 'cand_business_address': 'cand_address'})

s3 = s3.add_prefix('cand_')
s3 = s3.rename(columns={'cand_entity_id': 'candidate_entity_id', 'cand_business_name': 'cand_name', 'cand_business_address': 'cand_address'})


## 5. Load Dense Embedding Artifacts (Optional)

In [8]:
dense_cosine_available = False
dense_file = '../experiments/dense_candidate_pairs_train.tsv' if os.path.exists('../experiments/dense_candidate_pairs_train.tsv') else 'experiments/dense_candidate_pairs_train.tsv'
if os.path.exists(dense_file):
    dense_df = pd.read_csv(dense_file, sep='\t')
    if 'dense_cosine_similarity' in dense_df.columns:
        dense_cosine_available = True
        print("Loaded precomputed dense similarity scores.")


## 6. Pair Construction

In [10]:
if 'candidate_entity_id' not in candidates_df.columns and 'candidate_entity_ids' in candidates_df.columns:
    candidates_df['candidate_entity_id'] = candidates_df['candidate_entity_ids'].astype(str).str.split(',')
    candidates_df = candidates_df.explode('candidate_entity_id')

candidates_df['candidate_source'] = candidates_df['candidate_entity_id'].astype(str).str.split('-').str[0]

cand_s2 = candidates_df[candidates_df['candidate_source'] == 'S2']
cand_s3 = candidates_df[candidates_df['candidate_source'] == 'S3']

merged_s2 = cand_s2.merge(s1, on='source1_entity_id', how='left')
merged_s3 = cand_s3.merge(s1, on='source1_entity_id', how='left')

merged_s2 = merged_s2.merge(s2, on='candidate_entity_id', how='left')
merged_s3 = merged_s3.merge(s3, on='candidate_entity_id', how='left')

df_pairs = pd.concat([merged_s2, merged_s3], ignore_index=True)
print(f"Merged candidate pairs with text: {len(df_pairs)}")


Merged candidate pairs with text: 798962


## 7 - 12. Feature Generation
Generates Name, Address, Structured Address, Multilingual, Metadata, and Missingness features.

In [12]:
df_features = generate_all_features(df_pairs)
print(f"Generated features matrix shape: {df_features.shape}")


Computing string features for s1_name and cand_name with prefix 'name'...
Computing string features for s1_address and cand_address with prefix 'address'...
Computing structured address features...
Computing multilingual features...
Computing metadata features...
Computing missingness features...
Generated features matrix shape: (798962, 58)


## 13. Ground-Truth Labels

In [14]:
gt_file = '../dataset/train/train_ground_truth.tsv' if os.path.exists('../dataset/train/train_ground_truth.tsv') else 'dataset/train/train_ground_truth.tsv'
gt = pd.read_csv(gt_file, sep='\t')
gt['matched_entity_id'] = gt['matched_entity_ids'].astype(str).str.split(',')
gt = gt.explode('matched_entity_id')
gt['label_true'] = 1

df_features = df_features.merge(
    gt[['source1_entity_id', 'matched_entity_id', 'label_true']], 
    left_on=['source1_entity_id', 'candidate_entity_id'], 
    right_on=['source1_entity_id', 'matched_entity_id'], 
    how='left'
)
df_features['label'] = df_features['label_true'].fillna(0).astype(np.int8)
df_features = df_features.drop(columns=['label_true', 'matched_entity_id'], errors='ignore')

print(f"Total pairs: {len(df_features)}, Positive matches: {df_features['label'].sum()}")


Total pairs: 798962, Positive matches: 6169


## 14. Entity-Level Train / Validation Split
Splitting by `source1_entity_id` to prevent data leakage.

In [16]:
unique_s1 = np.array(list(df_features['source1_entity_id'].unique()))
train_s1, val_s1 = train_test_split(unique_s1, test_size=0.2, random_state=42)

train_s1_set = set(train_s1)
val_s1_set = set(val_s1)

train_df = df_features[df_features['source1_entity_id'].isin(train_s1_set)].copy()
val_df = df_features[df_features['source1_entity_id'].isin(val_s1_set)].copy()

print(f"Train pairs: {len(train_df)}, Validation pairs: {len(val_df)}")


Train pairs: 639204, Validation pairs: 159758


## 15 & 16. Feature Statistics & Distribution Analysis

In [18]:
non_feature_cols = ['source1_entity_id', 'candidate_entity_id', 'candidate_source', 'label', 'candidate_entity_ids']
feature_cols = [c for c in train_df.columns if c not in non_feature_cols and not c.startswith('s1_') and not c.startswith('cand_')]

stats = []
for c in feature_cols:
    s = {
        'feature': c,
        'dtype': str(train_df[c].dtype),
        'missing_pct': float(train_df[c].isna().mean()),
        'min': float(train_df[c].min()) if pd.api.types.is_numeric_dtype(train_df[c]) else None,
        'max': float(train_df[c].max()) if pd.api.types.is_numeric_dtype(train_df[c]) else None,
        'mean': float(train_df[c].mean()) if pd.api.types.is_numeric_dtype(train_df[c]) else None,
        'std': float(train_df[c].std()) if pd.api.types.is_numeric_dtype(train_df[c]) else None
    }
    stats.append(s)

exp_dir = '../experiments' if os.path.exists('../experiments') else 'experiments'
with open(os.path.join(exp_dir, 'feature_statistics.json'), 'w') as f:
    json.dump(stats, f, indent=4)

print(f"Computed statistics for {len(feature_cols)} features.")


Computed statistics for 45 features.


## 17. Correlation Analysis

In [20]:
numeric_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(train_df[c])]
if len(numeric_cols) > 0:
    corr_matrix = train_df[numeric_cols].corr()
    print("Top correlated feature pairs:")
    high_corr = []
    for i in range(len(numeric_cols)):
        for j in range(i+1, len(numeric_cols)):
            val = corr_matrix.iloc[i, j]
            if abs(val) > 0.85:
                high_corr.append((numeric_cols[i], numeric_cols[j], float(val)))
    print(high_corr[:10])


Top correlated feature pairs:
[('name_exact_match', 'name_compact_exact_match', 0.8632213061912117), ('name_length_ratio', 'name_length_difference', -0.9082092208947438), ('name_token_jaccard', 'name_token_overlap_s1', 0.9278952871949376), ('name_token_jaccard', 'name_token_overlap_candidate', 0.9438741743007177), ('name_char_4gram_jaccard', 'name_char_4gram_overlap', 0.9031241601598107), ('name_char_4gram_jaccard', 'name_tfidf_cosine', 1.0), ('name_char_4gram_overlap', 'name_tfidf_cosine', 0.9031241601598107), ('address_exact_match', 'address_compact_exact_match', 0.9825475733035215), ('address_token_jaccard', 'address_token_overlap_s1', 0.9665995202873308), ('address_token_jaccard', 'address_token_overlap_candidate', 0.9535709007743182)]


## 18. Manual Pair Inspection

In [22]:
pos_sample = train_df[train_df['label'] == 1].head(2)
neg_sample = train_df[train_df['label'] == 0].head(2)

print("Sample Positive Pair:")
print(pos_sample[['source1_entity_id', 'candidate_entity_id', 'name_edit_similarity', 'address_edit_similarity', 'label']])

print("\nSample Negative Pair:")
print(neg_sample[['source1_entity_id', 'candidate_entity_id', 'name_edit_similarity', 'address_edit_similarity', 'label']])


Sample Positive Pair:
    source1_entity_id candidate_entity_id  ...  address_edit_similarity  label
23       S1-100404750        S2-253678935  ...                 0.950000      1
130      S1-100627289        S2-594748318  ...                 0.233333      1

[2 rows x 5 columns]

Sample Negative Pair:
   source1_entity_id candidate_entity_id  ...  address_edit_similarity  label
24      S1-100404750        S2-180271999  ...                 0.105263      0
25      S1-100404750        S2-812728839  ...                 0.000000      0

[2 rows x 5 columns]


## 19. Feature Sanity Checks

In [24]:
assert not train_df[feature_cols].isna().any().any(), "Unexpected nulls in feature columns!"
assert 'source1_entity_id' not in feature_cols, "Entity ID leaked into feature columns!"
assert 'candidate_entity_id' not in feature_cols, "Candidate ID leaked into feature columns!"
print("All sanity checks passed successfully.")


All sanity checks passed successfully.


## 20. Save Training & Validation Feature Datasets

In [26]:
schema = {
    'features': feature_cols,
    'metadata': ['source1_entity_id', 'candidate_entity_id', 'candidate_source'],
    'target': 'label'
}
exp_dir = '../experiments' if os.path.exists('../experiments') else 'experiments'
with open(os.path.join(exp_dir, 'feature_schema.json'), 'w') as f:
    json.dump(schema, f, indent=4)

keep_cols = schema['metadata'] + schema['features'] + [schema['target']]
for c in schema['metadata']:
    train_df[c] = train_df[c].astype(str)
    val_df[c] = val_df[c].astype(str)

train_df[keep_cols].to_parquet(os.path.join(exp_dir, 'train_pair_features.parquet'), index=False)
val_df[keep_cols].to_parquet(os.path.join(exp_dir, 'validation_pair_features.parquet'), index=False)

print(f"Successfully saved train ({len(train_df)}) and validation ({len(val_df)}) feature datasets to Parquet!")


Successfully saved train (639204) and validation (159758) feature datasets to Parquet!


## 21. Findings
- Total candidate pairs feature-engineered.
- Computed comprehensive feature vector combining string, structured, script, metadata, and missingness metrics.
- Created clean entity-level train/validation split and persisted Parquet + schema artifacts.